In [10]:
import wandb
import numpy as np
import re

def extract_histogram_data(hist_dict):
    if not (isinstance(hist_dict, dict) and "_type" in hist_dict and hist_dict["_type"] == "histogram"):
        return None
    if "values" in hist_dict and isinstance(hist_dict["values"], (list, np.ndarray)):
        arr = np.array(hist_dict["values"])
        return np.abs(arr)
    elif "bins" in hist_dict and "counts" in hist_dict:
        bins = np.array(hist_dict['bins'])
        counts = np.array(hist_dict['counts'])
        if len(bins) != len(counts) + 1:
            return None
        centers = 0.5 * (bins[:-1] + bins[1:])
        abs_vals = np.abs(centers)
        return np.repeat(abs_vals, counts.astype(int))
    else:
        return None

def format_layer_name(lname):
    """
    Shorten layer names: e.g.,
    k_layers.0.linear -> k0
    m_layers.1.weight -> m1
    embedding.weight -> embedding
    """
    # Embedding
    if lname.startswith("embedding"):
        return "embedding"
    # Grouped layers like k_layers.3.linear or k_layers.3.weight
    m = re.match(r"([a-zA-Z]+)_layers\.(\d+)", lname)
    if m:
        group, idx = m.groups()
        return f"{group[0]}{idx}"
    # fallback: just use first part before dot
    return lname.split(".")[0]

def get_final_gradient_stats(run):
    latest = {}
    for row in run.scan_history():
        step = row.get("_step") or row.get("step")
        for key, val in row.items():
            if (
                key.startswith("gradients/")
                and key.endswith(".weights")
                and isinstance(val, dict) and val.get("_type") == "histogram"
            ):
                lname = key.replace("gradients/", "").replace(".weights", "")
                latest[lname] = (step, val)
    stats = []
    for lname, (step, hist) in latest.items():
        values = extract_histogram_data(hist)
        if values is not None and len(values) > 0:
            mean = values.mean()
            std = values.std()
            ratio = std / mean if mean != 0 else np.nan
            stats.append([format_layer_name(lname), mean, std, ratio])
    stats = sorted(stats, key=lambda x: x[0])
    return stats

def format_sci(val, digits=2):
    return f"{val:.{digits}e}"

def make_latex_table(stats):
    header = (
        "\\begin{table*}[!ht]\n"
        "\\centering\n"
        "\\caption{Final layerwise gradient statistics: mean, std, and std/mean for $|\\nabla_w L|$ (absolute gradient) at the end of training.}\n"
        "\\label{tab:gradient_stats}\n"
        "\\vskip 0.15in\n"
        "\\begin{center}\n"
        "\\begin{small}\n"
        "\\begin{sc}\n"
        "\\begin{tabular}{l|c|c|c}\n"
        "Layer & Mean & Std & Std/Mean \\\\\n\\hline\n"
    )
    body = ""
    for row in stats:
        lname, mean, std, ratio = row
        body += (
            f"{lname} & {format_sci(mean)} & {format_sci(std)} & {format_sci(ratio,3)} \\\\\n"
        )
    footer = (
        "\\hline\n"
        "\\end{tabular}\n"
        "\\end{sc}\n"
        "\\end{small}\n"
        "\\end{center}\n"
        "\\vskip -0.1in\n"
        "\\end{table*}\n"
    )
    return header + body + footer

def main():
    run_path = "sbuehrer-eth-z-rich/RDDLGN/i1cjqgzd"
    api = wandb.Api()
    run = api.run(run_path)
    stats = get_final_gradient_stats(run)
    latex_table = make_latex_table(stats)
    with open("gradient_stats_table.tex", "w") as f:
        f.write(latex_table)
    print("LaTeX table saved to gradient_stats_table.tex")
    print("----LaTeX table preview----\n")
    print(latex_table)

if __name__ == "__main__":
    main()

LaTeX table saved to gradient_stats_table.tex
----LaTeX table preview----

\begin{table*}[!ht]
\centering
\caption{Final layerwise gradient statistics: mean, std, and std/mean for $|\nabla_w L|$ (absolute gradient) at the end of training.}
\label{tab:gradient_stats}
\vskip 0.15in
\begin{center}
\begin{small}
\begin{sc}
\begin{tabular}{l|c|c|c}
Layer & Mean & Std & Std/Mean \\
\hline
k0 & 5.90e+04 & 4.68e+05 & 7.936e+00 \\
l0 & 1.02e+04 & 8.11e+04 & 7.931e+00 \\
m0 & 7.37e+04 & 5.85e+05 & 7.932e+00 \\
m1 & 4.83e+04 & 3.83e+05 & 7.936e+00 \\
m2 & 4.83e+04 & 3.83e+05 & 7.937e+00 \\
m3 & 4.83e+04 & 3.81e+05 & 7.884e+00 \\
m4 & 4.83e+04 & 3.83e+05 & 7.933e+00 \\
m5 & 1.20e+05 & 9.52e+05 & 7.936e+00 \\
m6 & 1.20e+05 & 9.52e+05 & 7.935e+00 \\
n0 & 8.82e+03 & 6.90e+04 & 7.819e+00 \\
p0 & 7.82e+04 & 6.20e+05 & 7.927e+00 \\
\hline
\end{tabular}
\end{sc}
\end{small}
\end{center}
\vskip -0.1in
\end{table*}

